### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ljubljana_breast_cancer",
    dataset_year="1988",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51P4M",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/14/breast+cancer.zip && unzip breast+cancer.zip breast-cancer.data && rm breast+cancer.zip && mkdir -p local-data-warehouse/ljubljana_breast_cancer && mv breast-cancer.data local-data-warehouse/ljubljana_breast_cancer/
""",
    # References
    academic_reference_bibtex="""@misc{Zwitter1988BreastCancer,
  author       = {Zwitter, Matjaz and Soklic, Milan},
  title        = {{Breast Cancer}},
  year         = {1988},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C51P4M}
}
""",
    academic_reference_bibtex_key="Zwitter1988BreastCancer",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We encode missing values as np.nan instead of "?".
- We reverse the discretization of several numeric features and make them integers again.
- We keep duplicates as they seem naturally occurring.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Class",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import re

columns = [
    "Class",
    "age",
    "menopause",
    "tumor-size",
    "inv-nodes",
    "node-caps",
    "deg-malig",
    "breast",
    "breast-quad",
    "irradiat"
]
df = pd.read_csv(dataset_mold.path / "breast-cancer.data", header=None, names=columns, na_values="?")
print("Loaded data shape:", df.shape)


def interval_midpoint(x):
    """
    Convert strings like '10-19' -> 14 (mean of boundaries, rounded),
    '?' / NaN -> NaN, already-numeric -> as-is.
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return x

    s = str(x).strip()
    if s in {"?", ""}:
        return np.nan

    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", s)
    if not m:
        return np.nan  # or return s if you prefer to keep non-interval values

    a = int(m.group(1))
    b = int(m.group(2))
    return (a + b) / 2

for c in ["age", "tumor-size", "inv-nodes"]:
    df[c] = df[c].map(interval_midpoint)

as_cat_type = ["Class", "menopause", "node-caps", "breast", "breast-quad", "irradiat"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (286, 10)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 286
Columns: 10
Use sampling: False (sample size: 286)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['tumor-size', 'inv-nodes', 'age', 'breast-quad', 'deg-malig', 'menopause', 'node-caps', 'irradiat', 'breast']
Rows remaining as candidates after top-9 filter: 39 (of 286)

#### Duplicate Report
Total duplicate rows: 14 (4.90% of dataset)
Duplicate rows ignoring target: 20 (6.99% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Class,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat
0,no-recurrence-events,44.5,premeno,22.0,1.0,no,2,right,left_up,no
1,recurrence-events,64.5,ge40,22.0,25.0,yes,3,left,left_low,yes
2,no-recurrence-events,44.5,premeno,47.0,1.0,no,2,left,left_low,yes
3,recurrence-events,44.5,premeno,32.0,1.0,no,3,right,right_up,no
4,recurrence-events,54.5,premeno,32.0,1.0,no,3,right,left_up,yes


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,node-caps,category,8.0,2.80,2.0,"no, yes"
1,breast-quad,category,1.0,0.35,5.0,"left_low, left_up, right_up, right_low, central"
2,Class,category,0.0,0.00,2.0,"no-recurrence-events, recurrence-events"
3,menopause,category,0.0,0.00,3.0,"premeno, ge40, lt40"
4,breast,category,0.0,0.00,2.0,"left, right"
5,irradiat,category,0.0,0.00,2.0,"no, yes"
6,age,float64,0.0,0.00,6.0,"54.5, 44.5, 64.5, 34.5, 74.5, 24.5"
7,tumor-size,float64,0.0,0.00,11.0,"32.0, 27.0, 22.0, 17.0, 12.0, 42.0, 37.0, 2.0, 52.0, 7.0"
8,inv-nodes,float64,0.0,0.00,7.0,"1.0, 4.0, 7.0, 10.0, 16.0, 13.0, 25.0"
9,deg-malig,int64,0.0,0.00,3.0,"2, 3, 1"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,286.0,51.143357,10.118183,24.5,74.5
tumor-size,286.0,26.405594,10.529649,2.0,52.0
inv-nodes,286.0,2.573427,3.451904,1.0,25.0
deg-malig,286.0,2.048951,0.738217,1.0,3.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column      rank                                    
Class       1     no-recurrence-events    201  70.28
            2        recurrence-events     85  29.72
breast      1                     left    152  53.15
            2                    right    134  46.85
breast-quad 1                 left_low    110  38.46
            2                  left_up     97  33.92
            3                 right_up     33  11.54
            4                right_low     24   8.39
            5                  central     21   7.34
irradiat    1                       no    218  76.22
            2                      yes     68  23.78
menopause   1                  premeno    150  52.45
            2                     ge40    129  45.10
            3                     lt40      7   2.45
node-caps   1                       no    222  77.62
            2                      yes     56  19.58
            3                     <NA>      8   2.80

In [8]:
# Target Distribution
target_df

,count,pct
Class,,
no-recurrence-events,201,70.28
recurrence-events,85,29.72


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to ljubljana_breast_cancer/019d5dc4-af45-7902-abe4-02e16ca4cc9e
019d5dc4-af45-7902-abe4-02e16ca4cc9e
50c486daf6e2c9a4fc9866832d9942c655805664f4c4ab000ba6867ea29316e4
